## Q1 – combined_text length at index 51

In [ ]:
from datasets import load_dataset

ds = load_dataset('csv', data_files='train.csv', split='train')

def make_combined(example):
    example['combined_text'] = str(example['prompt']) + ' ' + str(example['A'])
    return example

ds = ds.map(make_combined)
print('Q1 Answer:', len(ds[51]['combined_text']))

## Q2 – Vocab size

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
print('Q2 Answer:', tokenizer.vocab_size)

## Q3 – [SEP] token ID

In [ ]:
print('Q3 Answer:', tokenizer.sep_token_id)

## Q4 – input_ids shape

In [ ]:
encoded = tokenizer(
    ds['prompt'],
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)
print('Q4 Answer:', tuple(encoded['input_ids'].shape))

## Q5 – Attention head dim

In [ ]:
print('Q5 Answer:', 768 // 12)

## Q6 – last_hidden_state shape for row 0

In [ ]:
import torch
from transformers import AutoModel

model = AutoModel.from_pretrained('bert-base-uncased')
model.eval()

inputs_row0 = tokenizer(ds[0]['prompt'], return_tensors='pt')
with torch.no_grad():
    outputs_row0 = model(**inputs_row0)

print('Q6 Answer:', tuple(outputs_row0.last_hidden_state.shape))

## Q7 – Sum of first 5 CLS floats

In [ ]:
cls_vector = outputs_row0.last_hidden_state[0, 0, :]
first_5 = cls_vector[:5].tolist()
print('Q7 Answer:', round(sum(first_5), 4))

## Q8 – Attention weight [CLS]→fusion

In [ ]:
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
model_attn.eval()

test_str = 'Light-ion fusion is a technique.'
inputs_test = tokenizer(test_str, return_tensors='pt')

with torch.no_grad():
    outputs_test = model_attn(**inputs_test)

tokens = tokenizer.convert_ids_to_tokens(inputs_test['input_ids'][0].tolist())
fusion_idx = tokens.index('fusion')

attn_weight = outputs_test.attentions[-1][0, 0, 0, fusion_idx].item()
print('Q8 Answer:', round(attn_weight, 4))

## Q9 – Cosine sim prompt vs B, row 0

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

emb_prompt = st_model.encode(ds[0]['prompt'])
emb_B      = st_model.encode(str(ds[0]['B']))

print('Q9 Answer:', round(cos_sim(emb_prompt, emb_B).item(), 4))

## Q10 – MAP@3 TF-IDF vs MiniLM

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine

OPTIONS = ['A','B','C','D','E']
answers = ds['answer']

def ap_at_3(ranked, correct):
    for i, opt in enumerate(ranked):
        if opt == correct:
            return 1 / (i + 1)
    return 0.0

def map_at_3(preds, ans):
    return float(np.mean([ap_at_3(p, a) for p, a in zip(preds, ans)]))

# TF-IDF
tfidf_preds = []
for row in ds:
    docs = [str(row['prompt'])] + [str(row[o]) for o in OPTIONS]
    vec  = TfidfVectorizer().fit_transform(docs)
    sims = sk_cosine(vec[0:1], vec[1:])[0]
    tfidf_preds.append([OPTIONS[i] for i in np.argsort(sims)[::-1]][:3])

# MiniLM
all_p_embs = st_model.encode(ds['prompt'], batch_size=64, show_progress_bar=True)
all_o_embs = {o: st_model.encode([str(v) for v in ds[o]], batch_size=64) for o in OPTIONS}

mini_preds = []
for i in range(len(ds)):
    sims   = [cos_sim(all_p_embs[i], all_o_embs[o][i]).item() for o in OPTIONS]
    mini_preds.append([OPTIONS[j] for j in np.argsort(sims)[::-1]][:3])

print('Q10 MAP@3 MiniLM:', round(map_at_3(mini_preds, answers), 4))
print('Q10 TF-IDF miss & MiniLM hit:', sum(
    1 for i, a in enumerate(answers) if a not in tfidf_preds[i] and a in mini_preds[i]
))

## Q11 – Zero-shot Softmax, row 1

In [ ]:
from transformers import pipeline

zs = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

row1   = ds[1]
labels = [str(row1['A']), str(row1['B']), str(row1['C'])]

res_softmax = zs(row1['prompt'], candidate_labels=labels, multi_label=False)
sum_softmax = sum(res_softmax['scores'])

print('Q11 Top score:', round(res_softmax['scores'][0], 4))
print('Q11 Top label:', res_softmax['labels'][0])

## Q12 – Zero-shot Sigmoid, absolute diff

In [ ]:
res_sigmoid = zs(row1['prompt'], candidate_labels=labels, multi_label=True)
sum_sigmoid = sum(res_sigmoid['scores'])

print('Q12 Answer:', round(abs(sum_softmax - sum_sigmoid), 4))

## Q13 – Flan-T5-small generative QA

In [ ]:
gen = pipeline('text2text-generation', model='google/flan-t5-small')

r0 = ds[0]
input_str = (
    f"Question: {r0['prompt']}. "
    f"Is the correct answer A: {r0['A']} or B: {r0['B']}? "
    f"Answer with just the letter A or B."
)

output = gen(input_str, max_new_tokens=5)
print('Q13 Answer:', output[0]['generated_text'])